# slv_tickets
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · entidad Tickets de soporte |
| Propósito | bronze.support_tickets_raw → silver.tickets (SCD1, incluye los DELETE). El texto se conserva tal cual (emojis, textos largos): es insumo real para features y para el RAG. |
| Reglas | reglas_calidad.REGLAS["tickets"] |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun import TECNICAS, con_secuencia, leer_bronze, parametros_cdc
from reglas_calidad import reglas

# AUTO CDC: nombre actual de la API; apply_changes es el nombre anterior.
aplicar_cdc = getattr(dlt, "create_auto_cdc_flow", None) or dlt.apply_changes

## 2. Vista de limpieza `limpio_tickets`
Lee los cambios de bronze **como stream** (solo lo nuevo en cada corrida) y:
1. agrega el **orden de los cambios** (`_secuencia`: fecha efectiva, `_lsn`, `_seqval`);
2. **renombra** a snake_case en español y **normaliza** valores;
3. aplica las **reglas de calidad** en tres niveles: `fallar` (detiene el pipeline), `descartar` (no entra; la recoge la cuarentena) y `advertir` (entra y queda la métrica).

In [ ]:
@dlt.view(name="limpio_tickets", comment="Tickets de bronze renombrados y validados.")
@dlt.expect_all_or_fail(reglas("tickets", "fallar"))      # rompe el modelo → detiene el pipeline
@dlt.expect_all_or_drop(reglas("tickets", "descartar"))   # inválido → no entra (va a cuarentena)
@dlt.expect_all(reglas("tickets", "advertir"))            # mejorable → entra y queda la métrica
def limpio_tickets():
    df = con_secuencia(leer_bronze("support_tickets_raw"))
    return df.select(
        F.col("TicketID").alias("id_ticket"),
        F.col("CustomerID").alias("id_cliente"),
        F.col("OrderID").alias("id_pedido"),
        F.col("Subject").alias("asunto"),
        F.col("Body").alias("cuerpo"),
        F.lower("Priority").alias("prioridad"),
        F.lower("Status").alias("estado"),
        F.col("OpenedAt").alias("fecha_apertura"),
        F.col("ClosedAt").alias("fecha_cierre"),
        F.col("CreatedAt").alias("creado_en"),
        F.col("UpdatedAt").alias("actualizado_en"),
        F.col("Body").isNotNull().alias("tiene_cuerpo"),
        *TECNICAS)

## 3. Tabla(s) silver con AUTO CDC: `tickets`
AUTO CDC aplica los cambios **en su orden real** (`_secuencia`), se queda con el último por clave (resuelve duplicados y llegadas fuera de orden) y convierte los `D` de la fuente en **borrados**.
- **SCD1** → estado actual (una fila por clave).

In [ ]:
dlt.create_streaming_table(name="tickets", comment="Tickets de soporte: estado actual (SCD1) desde CDC de Azure SQL, incluidos los borrados.")
aplicar_cdc(target="tickets", source="limpio_tickets", **parametros_cdc("id_ticket"))
# Lo que no cumple una regla de descarte lo recoge slv_cuarentena.py.